# PNWER Project – Data Cleaning: StatCan Table 12-10-0099-01

**Source:** Statistics Canada, Table 12-10-0099-01 – *Merchandise imports and exports, customs-based, by HS section, Canada, provinces and territories, United States, states* (monthly).

**Scope:** British Columbia and Alberta ↔ Washington and Idaho, all HS sections, 1990-01 onward.

**Output:** one clean, tidy table (CSV + SQLite) that the dashboard can read directly.

**AI disclosure:** Code drafted with help from Claude (Anthropic). The team ran, checked, and is responsible for the results.

In [ ]:
import requests, zipfile, io
import sqlite3
import pandas as pd

**Why we download from the StatCan web service:**

Downloading by hand from the website works once, but the numbers are revised and new months are added every month. Pulling the table with code means anyone on the team can re-run this notebook and get the latest version in the same format.

**General steps:**

Ask the StatCan web service for the download link of the full table.

Download the zip file and read the main CSV inside it (the other file is metadata).

Look at the shape, date range, and column names before cleaning anything.

In [ ]:
#PART 1 - Download
# 1) Ask StatCan for the download link of the full table (table id: 12100099)
api = "https://www150.statcan.gc.ca/t1/wds/rest/getFullTableDownloadCSV/12100099/en"
zip_url = requests.get(api).json()["object"]

# 2) Download the zip and read the main CSV in chunks, keeping only our provinces and states.
#    Same columns as reading the whole file, but a few hundred MB of memory instead of several GB.
#    (The website's automated version, pipeline/corridors.py, fetches only the series it needs.)
provinces = ["British Columbia", "Alberta"]
states = ["Washington, State", "Idaho"]          # StatCan's exact labels
z = zipfile.ZipFile(io.BytesIO(requests.get(zip_url).content))
csv_name = [n for n in z.namelist() if "MetaData" not in n][0]
parts = []
for chunk in pd.read_csv(z.open(csv_name), low_memory=False, chunksize=500_000):
    parts.append(chunk[chunk["GEO"].str.strip().isin(provinces) & chunk["United States"].str.strip().isin(states)])
df = pd.concat(parts, ignore_index=True)


In [ ]:
#PART 2 - First look at the raw table
print(df.shape)                                        # (rows, columns)
print(df["REF_DATE"].min(), "->", df["REF_DATE"].max())  # date range
print(df["Trade"].unique())                            # trade types

In [ ]:
print(df.columns.tolist())

#Note on the raw table:
The full table has about 15.2 million rows and 17 columns, covering 1990-01 to 2026-09. It includes every province and territory, every U.S. state, and 22 HS items (21 sections + a total). Reading it in chunks keeps only our four jurisdictions' rows (77,616), so the notebook runs comfortably in Colab; the checks below still confirm the names and units.

*Repo copy: same steps as the original notebook; only Part 1 now reads the file in chunks. The website automates this as `pipeline/corridors.py`, which reproduces the 77,616 values and 7,542 zeros exactly.*


**Why we check names and units before filtering:**

If a name doesn't match exactly, the filter silently returns fewer rows and we wouldn't notice. For example, StatCan writes Washington as **"Washington, State"** (to separate it from Washington, D.C.), so filtering for "Washington" returns nothing.

The VALUE column is in **thousands** of dollars. We convert it to dollars, but only after checking that every row really uses the same unit.

In [ ]:
#PART 3 - Clean
# 0) Our scope
provinces = ["British Columbia", "Alberta"]
states = ["Washington, State", "Idaho"]          # StatCan's exact labels

# Cleaner names for the dashboard
state_names = {"Washington, State": "Washington"}

# 1) Remove stray spaces so name matching is reliable
for col in ["GEO", "United States", "Trade", "Harmonized System (HS) Sections"]:
    df[col] = df[col].astype(str).str.strip()

# 2) Checks: stop here if the units or names are not what we expect
assert set(df["SCALAR_FACTOR"].dropna().unique()) == {"thousands"}, "Unit changed - check before converting"
assert set(provinces) <= set(df["GEO"].unique()), "A province name was not found"
assert set(states) <= set(df["United States"].unique()), "A state name was not found"

# 3) Keep only our four jurisdictions
sub = df[df["GEO"].isin(provinces) & df["United States"].isin(states)]

# 4) Build the clean table in our standard format
clean = pd.DataFrame({
    "date": pd.to_datetime(sub["REF_DATE"], format="%Y-%m"),
    "reporter": sub["GEO"],                                     # Canadian province
    "partner": sub["United States"].replace(state_names),        # U.S. state
    "flow": sub["Trade"],                                        # Domestic exports / Total imports
    "hs_section_code": sub["Harmonized System (HS) Sections"].str.split(" - ").str[0],  # e.g. "IX"
    "hs_section_label": sub["Harmonized System (HS) Sections"],
    "value_cad": sub["VALUE"] * 1000,                           # thousands -> dollars (nominal)
    "currency": "CAD",
    "status_flag": sub["STATUS"],                                # StatCan quality flags, kept on purpose
    "symbol_flag": sub["SYMBOL"],
    "terminated_flag": sub["TERMINATED"],
    "series_id": sub["VECTOR"],                                  # StatCan series id, for traceability
    "source": "Statistics Canada, Table 12-10-0099-01",
    "retrieved_at": pd.Timestamp.today().normalize(),
})

# 5) Sort so each series reads in time order
clean = clean.sort_values(["reporter", "partner", "flow", "hs_section_code", "date"]).reset_index(drop=True)

In [ ]:
#PART 4 - Quality checks
# 1) Size and coverage
print("Rows:", len(clean))
print("Date range:", clean["date"].min().date(), "->", clean["date"].max().date())
print("Flows:", clean["flow"].unique().tolist())
print(clean.groupby(["reporter", "partner"]).size())     # expected: 441 months x 22 HS x 2 flows = 19,404 each

In [ ]:
# 2) Missing values and StatCan flags
print("Missing values:", clean["value_cad"].isna().sum())
print("Rows with a status flag:", clean["status_flag"].notna().sum())
print("Zero values:", (clean["value_cad"] == 0).sum())

In [ ]:
# 3) Do the HS sections add up to the total?
is_total = clean["hs_section_code"].str.startswith("Total")
keys = ["date", "reporter", "partner", "flow"]

sections = clean[~is_total].groupby(keys)["value_cad"].sum()
totals = clean[is_total].set_index(keys)["value_cad"]

print("Max gap between total and sum of sections:", (totals - sections).abs().max())   # small gap = rounding

In [ ]:
# 4) Where are the zeros?
zeros = clean[clean["value_cad"] == 0]
print(zeros.groupby(["reporter", "partner"]).size())
print(zeros["hs_section_code"].value_counts().head(8))

#Note on data quality:
The clean table has 77,616 rows: 19,404 for each province–state pair, which is exactly 441 months × 22 HS items × 2 flows, so no month or section is missing. There are no missing values and no StatCan status flags.

About 10% of rows (7,542) are zero. They are concentrated in low-volume pairs (Alberta–Idaho: 3,339; BC–Washington: only 147) and in minor HS sections (arms and ammunition, footwear, jewellery, works of art). The HS sections add up to the reported totals within CAD 5,000, which is consistent with rounding in the source (values are published in thousands). For these reasons, we read the zeros as "no trade" or "trade below about CAD 500", not as hidden data, and we keep them as zero.

#Note on trade types:
At the province–state level, the table only has **Domestic exports** and **Total imports**. "Total exports" and "Re-exports" only exist at the national level. This is helpful: domestic exports measure goods produced in the province, so there is no risk of double counting re-exported goods.

Things to remember when using this table:
- Values are **nominal CAD**, not adjusted for inflation or prices. For Alberta, falling oil prices can look like a tariff effect.
- There are **no quantities**, only values.
- The partner state is the declared destination/origin, not necessarily where the goods are used.
- Recent months can be revised by StatCan, so `retrieved_at` records when we pulled the data.

**Why we save to Google Drive:**

Files saved in Colab's default folder (`/content/`) are deleted when the session ends. Saving to Drive keeps them, and the SQLite file can be read directly by the dashboard.

In [ ]:
#PART 5 - Save
# 1) Connect Google Drive (asks for permission the first time)
from google.colab import drive
drive.mount("/content/drive")

# 2) Folder for the project data
out_dir = "/content/drive/MyDrive/PNWER/data"
!mkdir -p "$out_dir"

# 3) Save as CSV and as a SQLite table
clean.to_csv(f"{out_dir}/statcan_12100099_bc_ab_wa_id.csv", index=False)

with sqlite3.connect(f"{out_dir}/pnwer_trade.db") as con:
    clean.to_sql("trade_province_state_hs", con, if_exists="replace", index=False)

print("Saved to", out_dir)